In [75]:
from transformers import pipeline

model = "Qwen/Qwen2.5-1.5B"

generator = pipeline(
    "text-generation",
     model=model,
     max_length=None,
     max_new_tokens=150, 
     do_sample=True,
     return_full_text=False
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [76]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from app.app import load_resources

In [77]:
documents, bm25 = load_resources()

In [78]:
from src.semantic import create_faiss_index

vectorstore = create_faiss_index(documents, 10000, reload_index=False)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [79]:
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)

In [104]:
SYSTEM_PROMPT = """
    You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible."""

def build_prompt(query, context):
    return f"""{SYSTEM_PROMPT}

context:
{context}

question: 
{query}

Recommend ONE product using the context.
Do not add additional explanations or repeat the prompt.
Stop after the recommendation.

Return the answer exactly in this format:

Product Title:
Product ASIN:
Product Rating:
Product Review:
Reason for Recommendation: Write 2 natural sentences describing the product’s key benefits using evidence from the review and rating.

END
"""

In [105]:
def build_context(docs):
    return "\n\n".join(
        f"Product ASIN: {doc.metadata.get('asin')}\n"
        f"Product Title: {doc.metadata.get('product_title')}\n"
        f"Product Rating: {doc.metadata.get('product_rating')}\n"
        f"Product Review: {doc.metadata.get('product_review')}\n"
        for doc in docs
    )

In [106]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

format_context = RunnableLambda(build_context)
def prompt_builder(inputs):
    return build_prompt(inputs["input"], inputs["context"])

prompt = RunnableLambda(prompt_builder)


rag_chain = (
    {
        "context": retriever | format_context,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [107]:
query = "Moisturizing shampoo for thick curly hair"

response = rag_chain.invoke(query)

print(response)

Product Title: Just For Me Curl Peace Ultimate Detangling Shampoo
Product ASIN: B08MBC424Z
Product Rating: 5.0
Product Review: This product works great if you have thick curly hair.
Reason for Recommendation: Although the reviewer does not express enthusiasm, the product is highly recommended for its effectiveness in thick curly hair, and it is currently the author's favorite choice.
